# Chapter 12 Lab: Vision: Learning from Images (`ch07`)

In this lab, we'll build intuition about how convolution works, why pooling helps robustness, how augmentation improves generalization, and why shortcuts in data can fool us. All from scratch with numpy and matplotlib.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Seed for reproducibility
np.random.seed(42)

## 3. Convolution by Hand

Let's manually apply convolution to a simple binary image. We'll use numpy to compute the dot products, then visualize the input, kernels, and output feature maps.

In [ ]:
# Create a 7x7 binary image with a vertical bar
image = np.zeros((7, 7), dtype=np.float32)
image[:, 3] = 1  # Vertical bar in the middle column

# Define edge detection kernels
vertical_kernel = np.array([[-1, 0, 1], 
                             [-2, 0, 2], 
                             [-1, 0, 1]], dtype=np.float32)
horizontal_kernel = np.array([[-1, -2, -1], 
                               [0, 0, 0], 
                               [1, 2, 1]], dtype=np.float32)

def convolve_2d(image, kernel):
    """Apply convolution: slide kernel over image, compute dot products."""
    h, w = image.shape
    kh, kw = kernel.shape
    
    # Output size: (height - kernel_height + 1) x (width - kernel_width + 1)
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)
    
    for i in range(out_h):
        for j in range(out_w):
            # Extract patch and compute element-wise product sum
            patch = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(patch * kernel)
    
    return output

# Apply convolution
vertical_features = convolve_2d(image, vertical_kernel)
horizontal_features = convolve_2d(image, horizontal_kernel)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Row 1: Vertical edge detection
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Input Image\n(Vertical Bar)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(vertical_kernel, cmap='RdBu', norm=Normalize(vmin=-2, vmax=2))
axes[0, 1].set_title('Vertical Edge\nKernel (Sobel-X)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

im0 = axes[0, 2].imshow(vertical_features, cmap='hot')
axes[0, 2].set_title('Vertical Feature Map\n(Strong at edges)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im0, ax=axes[0, 2], fraction=0.046)

# Row 2: Horizontal edge detection
axes[1, 0].imshow(image, cmap='gray')
axes[1, 0].set_title('Input Image', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(horizontal_kernel, cmap='RdBu', norm=Normalize(vmin=-2, vmax=2))
axes[1, 1].set_title('Horizontal Edge\nKernel (Sobel-Y)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

im1 = axes[1, 2].imshow(horizontal_features, cmap='hot')
axes[1, 2].set_title('Horizontal Feature Map\n(Mostly zero)', fontsize=12, fontweight='bold')
axes[1, 2].axis('off')
plt.colorbar(im1, ax=axes[1, 2], fraction=0.046)

plt.tight_layout()
plt.savefig('convolution_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print("Convolution insight: The vertical edge kernel activates strongly where vertical edges exist.")
print(f"Max vertical feature: {vertical_features.max():.1f}")
print(f"Max horizontal feature: {horizontal_features.max():.1f}")

## 4. Pooling

Apply max pooling and average pooling to reduce spatial dimensions while preserving important activations.

In [ ]:
def max_pool_2d(feature_map, pool_size=2):
    """Apply max pooling: take max in each pool_size x pool_size window."""
    h, w = feature_map.shape
    out_h, out_w = h // pool_size, w // pool_size
    output = np.zeros((out_h, out_w), dtype=np.float32)
    
    for i in range(out_h):
        for j in range(out_w):
            pool_region = feature_map[i*pool_size:(i+1)*pool_size, 
                                       j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(pool_region)
    
    return output

def avg_pool_2d(feature_map, pool_size=2):
    """Apply average pooling: take mean in each pool_size x pool_size window."""
    h, w = feature_map.shape
    out_h, out_w = h // pool_size, w // pool_size
    output = np.zeros((out_h, out_w), dtype=np.float32)
    
    for i in range(out_h):
        for j in range(out_w):
            pool_region = feature_map[i*pool_size:(i+1)*pool_size, 
                                       j*pool_size:(j+1)*pool_size]
            output[i, j] = np.mean(pool_region)
    
    return output

# Apply pooling to the vertical feature map
maxpool = max_pool_2d(vertical_features, pool_size=2)
avgpool = avg_pool_2d(vertical_features, pool_size=2)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

im0 = axes[0].imshow(vertical_features, cmap='hot')
axes[0].set_title('Feature Map\n(5x5)', fontsize=12, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(maxpool, cmap='hot')
axes[1].set_title('Max Pooling (2x2)\n(2x2)', fontsize=12, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(avgpool, cmap='hot')
axes[2].set_title('Average Pooling (2x2)\n(2x2)', fontsize=12, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('pooling_demo.png', dpi=100, bbox_inches='tight')
plt.show()

print("Pooling insight: Both max and average pooling reduce spatial size.")
print(f"Max pooling preserves peak activations. Shape: {vertical_features.shape} → {maxpool.shape}")
print(f"Avg pooling is smoother. Max values: {maxpool.max():.1f} vs {avgpool.max():.1f}")

## 5. Translation Sensitivity and Robustness

Shift the vertical bar by 2 pixels. Show that convolution output shifts (equivariance), but pooling is more robust to small shifts.

In [ ]:
# Create a shifted version of the image
image_shifted = np.zeros((7, 7), dtype=np.float32)
image_shifted[:, 5] = 1  # Bar shifted 2 positions right

# Apply convolution and pooling to both
vertical_features_orig = convolve_2d(image, vertical_kernel)
vertical_features_shifted = convolve_2d(image_shifted, vertical_kernel)

maxpool_orig = max_pool_2d(vertical_features_orig, pool_size=2)
maxpool_shifted = max_pool_2d(vertical_features_shifted, pool_size=2)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Original
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

im00 = axes[0, 1].imshow(vertical_features_orig, cmap='hot')
axes[0, 1].set_title('Convolution Output', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im00, ax=axes[0, 1], fraction=0.046)

im01 = axes[0, 2].imshow(maxpool_orig, cmap='hot')
axes[0, 2].set_title('After Max Pooling', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im01, ax=axes[0, 2], fraction=0.046)

# Shifted by 2 pixels
axes[1, 0].imshow(image_shifted, cmap='gray')
axes[1, 0].set_title('Shifted Image (+2 pixels)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

im10 = axes[1, 1].imshow(vertical_features_shifted, cmap='hot')
axes[1, 1].set_title('Convolution Output\n(also shifted)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im10, ax=axes[1, 1], fraction=0.046)

im11 = axes[1, 2].imshow(maxpool_shifted, cmap='hot')
axes[1, 2].set_title('After Max Pooling\n(similar!)', fontsize=12, fontweight='bold')
axes[1, 2].axis('off')
plt.colorbar(im11, ax=axes[1, 2], fraction=0.046)

plt.tight_layout()
plt.savefig('translation_robustness.png', dpi=100, bbox_inches='tight')
plt.show()

print("Equivariance & Robustness:")
print(f"Convolution is equivariant: shifting input shifts output.")
print(f"Pooled features similarity:")
print(f"  Original max pool max: {maxpool_orig.max():.1f}")
print(f"  Shifted max pool max: {maxpool_shifted.max():.1f}")
print(f"  → Pooling provides robustness to small translations.")

## 6. Augmentation Experiment

Train on clean handwritten digits, test on shifted versions. Then augment training with shifts and see improvement.

In [ ]:
# Load sklearn digits dataset
digits = load_digits()
X, y = digits.data, digits.target

# Split into train/test (80/20)
n_train = int(0.8 * len(X))
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

# Create a shifted test set (roll pixels by 1 position)
X_test_shifted = np.array([np.roll(img.reshape(8, 8), shift=1, axis=1).flatten() 
                            for img in X_test])

# Scenario 1: Train on clean, test on clean and shifted
model_clean = LogisticRegression(max_iter=1000, random_state=42)
model_clean.fit(X_train, y_train)

acc_clean_on_clean = accuracy_score(y_test, model_clean.predict(X_test))
acc_clean_on_shifted = accuracy_score(y_test, model_clean.predict(X_test_shifted))

# Scenario 2: Augment training with shifted versions, then test
X_train_augmented = X_train.copy()
y_train_augmented = y_train.copy()

# Add shifted versions (roll by 1, 2, -1, -2 positions)
for shift in [1, 2, -1, -2]:
    X_shifted_aug = np.array([np.roll(img.reshape(8, 8), shift=shift, axis=1).flatten() 
                              for img in X_train])
    X_train_augmented = np.vstack([X_train_augmented, X_shifted_aug])
    y_train_augmented = np.hstack([y_train_augmented, y_train])

model_aug = LogisticRegression(max_iter=1000, random_state=42)
model_aug.fit(X_train_augmented, y_train_augmented)

acc_aug_on_clean = accuracy_score(y_test, model_aug.predict(X_test))
acc_aug_on_shifted = accuracy_score(y_test, model_aug.predict(X_test_shifted))

# Visualize
scenarios = ['Clean Only\non Clean', 'Clean Only\non Shifted', 
             'Augmented\non Clean', 'Augmented\non Shifted']
accuracies = [acc_clean_on_clean, acc_clean_on_shifted, acc_aug_on_clean, acc_aug_on_shifted]
colors = ['green', 'orange', 'green', 'blue']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(scenarios, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Effect of Data Augmentation on Robustness', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.0])
ax.axhline(y=0.95, color='red', linestyle='--', linewidth=1, label='95% target')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('augmentation_impact.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n=== Augmentation Results ===")
print(f"Without augmentation:")
print(f"  Clean test accuracy: {acc_clean_on_clean:.4f}")
print(f"  Shifted test accuracy: {acc_clean_on_shifted:.4f}")
print(f"  Drop: {(acc_clean_on_clean - acc_clean_on_shifted):.4f}")
print(f"\nWith augmentation (shifts: ±1, ±2):")
print(f"  Clean test accuracy: {acc_aug_on_clean:.4f}")
print(f"  Shifted test accuracy: {acc_aug_on_shifted:.4f}")
print(f"  Drop: {(acc_aug_on_clean - acc_aug_on_shifted):.4f}")
print(f"\nImprovement on shifted data: {(acc_aug_on_shifted - acc_clean_on_shifted):.4f} ({(acc_aug_on_shifted - acc_clean_on_shifted)*100:.1f}%)")

## 7. Shortcut Detection

Generate synthetic data where one class has a systematic shortcut: a white corner pixel. Show the model exploits this, and how removing it reveals poor true generalization.

In [ ]:
# Generate synthetic images with a shortcut
def generate_shortcut_data(n_samples_per_class=100, seed=42):
    np.random.seed(seed)
    X, y = [], []
    
    for label in [0, 1]:
        for _ in range(n_samples_per_class):
            # Random noise image
            img = np.random.rand(8, 8)
            
            # Add shortcut: class 0 always has white pixel at (0,0)
            if label == 0:
                img[0, 0] = 1.0  # White corner pixel
            # class 1 has dark corner
            else:
                img[0, 0] = 0.0
            
            X.append(img.flatten())
            y.append(label)
    
    return np.array(X), np.array(y)

X_shortcut, y_shortcut = generate_shortcut_data(n_samples_per_class=100, seed=42)

# Split into train/test (80/20)
n_train_sc = int(0.8 * len(X_shortcut))
X_train_sc, X_test_sc = X_shortcut[:n_train_sc], X_shortcut[n_train_sc:]
y_train_sc, y_test_sc = y_shortcut[:n_train_sc], y_shortcut[n_train_sc:]

# Train on data with shortcut
model_with_shortcut = LogisticRegression(max_iter=1000, random_state=42)
model_with_shortcut.fit(X_train_sc, y_train_sc)
acc_with_shortcut = accuracy_score(y_test_sc, model_with_shortcut.predict(X_test_sc))

# Now remove the shortcut from test data (set corner pixel to 0.5 for both)
X_test_no_shortcut = X_test_sc.copy()
X_test_no_shortcut[:, 0] = 0.5  # Remove the distinguishing feature
acc_no_shortcut = accuracy_score(y_test_sc, model_with_shortcut.predict(X_test_no_shortcut))

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Example class 0 image (with shortcut)
sample_0 = X_shortcut[0].reshape(8, 8)
axes[0, 0].imshow(sample_0, cmap='gray')
axes[0, 0].set_title('Class 0 Example\n(White corner)', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

# Example class 1 image (without shortcut)
sample_1 = X_shortcut[110].reshape(8, 8)
axes[0, 1].imshow(sample_1, cmap='gray')
axes[0, 1].set_title('Class 1 Example\n(Dark corner)', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

# Coefficients visualization (first pixel vs others)
coef = model_with_shortcut.coef_[0]
axes[0, 2].bar(['Corner\n(Pixel 0)', 'Other Pixels\n(Mean)'], 
               [np.abs(coef[0]), np.mean(np.abs(coef[1:]))],
               color=['red', 'blue'], alpha=0.7, edgecolor='black')
axes[0, 2].set_ylabel('|Coefficient|', fontsize=11, fontweight='bold')
axes[0, 2].set_title('Model Learns Shortcut\n(Corner >> Others)', fontsize=11, fontweight='bold')
axes[0, 2].grid(axis='y', alpha=0.3)

# Shortcut present in test
sample_0_test = X_test_sc[0].reshape(8, 8)
axes[1, 0].imshow(sample_0_test, cmap='gray')
axes[1, 0].set_title('Test with Shortcut', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

# Shortcut removed from test
sample_0_no_sc = X_test_no_shortcut[0].reshape(8, 8)
axes[1, 1].imshow(sample_0_no_sc, cmap='gray')
axes[1, 1].set_title('Test without Shortcut', fontsize=11, fontweight='bold')
axes[1, 1].axis('off')

# Accuracy comparison
scenarios_sc = ['With Shortcut', 'Without Shortcut']
accs_sc = [acc_with_shortcut, acc_no_shortcut]
colors_sc = ['green', 'red']
bars = axes[1, 2].bar(scenarios_sc, accs_sc, color=colors_sc, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 2].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[1, 2].set_title('Accuracy Collapse Without Shortcut', fontsize=11, fontweight='bold')
axes[1, 2].set_ylim([0, 1.0])
axes[1, 2].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Random')
axes[1, 2].grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accs_sc):
    height = bar.get_height()
    axes[1, 2].text(bar.get_x() + bar.get_width()/2., height,
                   f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('shortcut_detection.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n=== Shortcut Detection ===")
print(f"Test accuracy WITH shortcut: {acc_with_shortcut:.4f}")
print(f"Test accuracy WITHOUT shortcut: {acc_no_shortcut:.4f}")
print(f"\nAccuracy drop: {(acc_with_shortcut - acc_no_shortcut):.4f} ({(acc_with_shortcut - acc_no_shortcut)*100:.1f}%)")
print(f"\n⚠️ Lesson: A model that achieves 100% accuracy may be exploiting\n   spurious correlations (shortcuts) rather than learning true patterns.")
print(f"\nAlways inspect what features your model relies on!")

## What to Notice

1. **Convolution detects local patterns**: A vertical edge kernel activates strongly where vertical transitions occur. Different kernels detect different patterns.

2. **Pooling provides robustness**: Both max and average pooling compress spatial information. Max pooling is sparse and preserves peak activations; average pooling is smoother. Small translations in the input cause shifts in the feature map, but pooling dampens this sensitivity.

3. **Translation sensitivity without pooling**: The raw convolution output is equivariant—it shifts when the input shifts. This can hurt generalization if your test data has shifted objects.

4. **Data augmentation improves robustness**: By training on shifted versions of images, the model learns to handle variations. Augmentation isn't about having more data; it's about teaching the model to be robust to realistic variations.

5. **Shortcuts fool aggregate metrics**: A model trained on data with spurious correlations (shortcuts) can achieve very high accuracy on in-distribution test data. But remove the shortcut, and performance collapses. High accuracy ≠ good generalization. Always ask: *What is my model actually using to make predictions?*

**Key Takeaway**: Vision models learn hierarchies of features through convolution and pooling. Augmentation and careful inspection of shortcuts are essential for real-world robustness.